# Hotel Recognition for Unseen Properties — Run-All (Colab GPU)

End-to-end run of the `hotelret` pipeline on a Colab **GPU** runtime. Produces the
real numbers for the paper's Experiment section.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Order of operations:
1. Setup (clone repo, install, check GPU)
2. Get dataset metadata
3. **Feasibility gate** — liveness + per-hotel clustering (STOP here if it fails)
4. Build the 1,000-hotel subset + unseen-hotel split
5. Download + resize images (resumable, ~20 min)
6. Extract embeddings: DINOv2, CLIP, SSCD (cached)
7. Evaluate: seen vs unseen, per encoder → the headline table
8. Extra arms: resolution ablation, duplicate detection
9. Save everything to `results/` and (optionally) Google Drive

Every result cell prints numbers you paste straight into the paper. Nothing is
fabricated — if a step is skipped, its table stays empty.

## 1. Setup

In [ ]:
# GPU check first — stop early if the runtime is CPU-only
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "NO GPU")
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then rerun."
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| device", torch.cuda.get_device_name(0))

In [ ]:
# Clone your repo. EDIT this URL to your GitHub once pushed; falls back to a local copy.
REPO_URL = "https://github.com/YOUR_USERNAME/hotel-retrieval.git"  # <-- EDIT
import os
from pathlib import Path

if not Path("hotel-retrieval").exists():
    r = os.system(f"git clone -q {REPO_URL}")
    if r != 0:
        print("Clone failed (repo not pushed yet?). Upload the repo folder to Colab")
        print("via the Files panel, or set REPO_URL correctly, then rerun.")
os.chdir("hotel-retrieval")
print("cwd:", Path.cwd())

In [ ]:
# Install: the package + the ML extras (torch is already on Colab)
!pip install -q -e . 2>&1 | tail -2
!pip install -q open_clip_torch timm faiss-cpu 2>&1 | tail -2
print("installed")

## 2. Dataset metadata

The metadata (14 MB) ships inside the official repo — no image download yet.

In [ ]:
import os, tarfile
from pathlib import Path

META_PARENT = Path("external/Hotels-50K")
if not META_PARENT.exists():
    os.system("git clone --depth 1 -q https://github.com/GWUvision/Hotels-50K.git external/Hotels-50K")
ds = META_PARENT / "input" / "dataset"
if not ds.exists():
    with tarfile.open(META_PARENT / "input" / "dataset.tar.gz") as t:
        t.extractall(META_PARENT / "input")

METADATA = str(ds)
from hotelret import data
import json
print(json.dumps(data.summarize(METADATA), indent=2))

## 3. Feasibility gate  ⛔

This is the go/no-go. It probes link liveness **and** tests whether losses cluster
by hotel. If the verdict is STOP, do not download — switch to the Kaggle fallback
(see the note at the bottom). This takes ~3–5 min.

In [ ]:
from hotelret import audit

train = data.load_train(METADATA)
test = data.load_test(METADATA)
per = data.images_per_hotel(train)
cand = [h for h in test.hotel_id.unique() if per.get(h, 0) >= 15]

sources = audit.probe_sources(train, n_per_source=300)
for s, v in sources.items():
    print(f"{s:16s} {v['rate']:.1%} alive")

clustering = audit.probe_clustering(train, cand[:2000], n_hotels=40)
print("\npooled survival    :", clustering["pooled_survival"])
print("overdispersion     :", clustering["overdispersion_ratio"], "x binomial")
print("hotels < 5 images  :", clustering["hotels_below_5_images"], "/", clustering["hotels_probed"])

verdict = audit.verdict(sources, clustering)
print("\n=== VERDICT ===\n" + verdict)

Path("results").mkdir(exist_ok=True)
Path("results/audit.json").write_text(json.dumps(
    {"sources": sources, "clustering": clustering, "verdict": verdict}, indent=2))

## 4. Build the subset + unseen-hotel split

In [ ]:
from hotelret import manifest

gallery, query, unseen = manifest.build(
    METADATA, n_hotels=1000, cap=40, min_train=15, unseen_frac=0.20, seed=0)
summary = manifest.write(gallery, query, unseen, "data/manifest")
print(json.dumps(summary, indent=2))

## 5. Download + resize images

Resumable — rerun to pick up where a dropped session left off. Expect ~20 min for
~26k images. `ok`/`cached` = success; `http_404` etc. = dead link (expected, that's
the decay we report).

In [ ]:
from hotelret import download
counts = download.run("data/manifest/gallery.csv", out_dir="data/images/gallery",
                      target=256, workers=24)
print("gallery:", counts)
counts_q = download.run("data/manifest/queries.csv", out_dir="data/images/query",
                        target=256, workers=24)
print("query:", counts_q)

In [ ]:
# how many actually landed, and drop hotels that fell under 5 gallery images
from pathlib import Path
import pandas as pd

got = pd.DataFrame({"path": [str(p) for p in Path("data/images/gallery").rglob("*.jpg")]})
got["hotel_id"] = got.path.map(lambda p: Path(p).parent.name)
counts = got.groupby("hotel_id").size()
usable = counts[counts >= 5].index
print(f"downloaded {len(got):,} gallery images across {counts.size} hotels")
print(f"hotels with >=5 images (kept): {len(usable)}")
Path("results/download_stats.json").write_text(json.dumps({
    "gallery_downloaded": int(len(got)),
    "hotels_with_images": int(counts.size),
    "hotels_kept_ge5": int(len(usable)),
}, indent=2))

## 6. Extract embeddings

One `.npy` per encoder, cached. DINOv2 and CLIP download weights automatically;
SSCD needs its standalone TorchScript file (one wget below). This is the main GPU
step — a few minutes per model on a T4.

In [ ]:
# SSCD standalone weights (ResNet-50, 512-dim) — the recommended default model
import urllib.request
from pathlib import Path
w = Path("external/sscd_disc_mixup.torchscript.pt")
if not w.exists():
    url = "https://dl.fbaipublicfiles.com/sscd-copy-detection/sscd_disc_mixup.torchscript.pt"
    print("downloading SSCD weights ...")
    urllib.request.urlretrieve(url, w)
print("SSCD weights:", (w.stat().st_size // 1_000_000), "MB" if w.exists() else "MISSING")

In [ ]:
from hotelret import embed
print("available encoders:", embed.available())

# embed BOTH gallery and query images together so vectors share one file per model
IMG_ROOT = "data/images"   # contains gallery/ and query/
for model in ["dinov2", "clip", "sscd"]:
    out = f"data/embeddings/{model}.npy"
    if Path(out).exists():
        print(f"{model}: cached"); continue
    print(f"embedding with {model} ...")
    shape = embed.extract(IMG_ROOT, model, out, batch_size=64)
    print(f"  {model}: {shape[0]} vectors x {shape[1]} dims")

## 7. Evaluate — the headline table

Top-K retrieval accuracy, **seen vs unseen** hotels, per encoder. The seen→unseen
drop, and whether the ranking of encoders changes, is the paper's main result.

In [ ]:
from hotelret import evaluate
import pandas as pd

rows = []
for model in ["dinov2", "clip", "sscd"]:
    emb = f"data/embeddings/{model}.npy"
    if not Path(emb).exists():
        print(f"skip {model} (no embeddings)"); continue
    for split in ["seen", "unseen"]:
        r = evaluate.evaluate(emb, "data/manifest", split)
        r["model"] = model
        rows.append(r)

tab = pd.DataFrame(rows)[["model", "split", "top1", "top10", "top100",
                          "n_query", "n_gallery"]]
print(tab.to_string(index=False))
tab.to_csv("results/main_results.csv", index=False)

# seen->unseen drop, the number the paper leads with
print("\n--- seen -> unseen top1 drop ---")
for m in tab.model.unique():
    s = tab[(tab.model==m)&(tab.split=='seen')].top1.values
    u = tab[(tab.model==m)&(tab.split=='unseen')].top1.values
    if len(s) and len(u):
        print(f"  {m:8s} {s[0]:.3f} -> {u[0]:.3f}  (drop {s[0]-u[0]:+.3f})")

In [ ]:
# a simple bar chart of top1 seen vs unseen
import matplotlib.pyplot as plt
import numpy as np

models = tab.model.unique()
seen = [tab[(tab.model==m)&(tab.split=='seen')].top1.values[0] for m in models]
unseen = [tab[(tab.model==m)&(tab.split=='unseen')].top1.values[0] for m in models]
x = np.arange(len(models)); w = 0.35
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(x-w/2, seen, w, label="seen")
ax.bar(x+w/2, unseen, w, label="unseen")
ax.set_xticks(x); ax.set_xticklabels(models); ax.set_ylabel("Top-1 accuracy")
ax.set_title("Seen vs unseen hotels"); ax.legend()
plt.tight_layout(); plt.savefig("results/fig_seen_vs_unseen.png", dpi=150)
plt.show()

## 8. Extra experiments (optional)

Two cheap arms once embeddings are cached: (a) does shrinking the resolution gap
close the accuracy gap, and (b) the duplicate-listing threshold. Run if time allows.

In [ ]:
# (a) Resolution ablation: re-embed queries downsized to gallery scale, compare.
# Requires re-running embed on a downscaled query folder — sketch left as a TODO
# so you can decide whether to spend the GPU time.
print("Resolution ablation: downscale data/images/query to ~112px short side,")
print("re-embed as e.g. dinov2_lowres.npy, and re-run evaluate. Fill in if time.")

In [ ]:
# (b) Duplicate-listing detection with SSCD.
# SSCD paper: cosine sim > 0.75 => copy at ~90% precision (DISC benchmark).
# Build synthetic near-duplicates with AugLy, then measure precision/recall.
try:
    import augly.image as imaugs
    print("AugLy ready — generate transformed copies of gallery images and test")
    print("SSCD cosine >= 0.75 as the copy threshold.")
except ImportError:
    print("pip install augly to run the duplicate-detection arm.")

## 9. Save results

In [ ]:
# everything the paper needs is in results/. Optionally copy to Drive.
import os
for f in sorted(Path("results").glob("*")):
    print(f, f.stat().st_size, "bytes")

SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.system("cp -r results /content/drive/MyDrive/hotel-retrieval-results")
    print("copied to Drive")

## If the feasibility gate said STOP

Switch the data source to Kaggle **Hotel-ID 2022 (FGVC9)**, which ships real image
files (no decay):

```python
!pip install -q kaggle
# upload kaggle.json (Account > Create New API Token), then:
!kaggle competitions download -c hotel-id-to-combat-human-trafficking-2022-fgvc9
```

The embed/evaluate steps are unchanged; only the manifest/download source differs.
You lose the occlusion arm (Kaggle has no occlusion variants) and keep everything
else.